# Machine health: learn before training
Fully synthetic data. Run each cell, inspect its output and write your observations. Never use future outcome columns as inputs. Notebook expects its current folder to contain `data/` and `train.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
assert (ROOT/'data/telemetry_raw.csv').exists(), 'Open Jupyter in the machine-health-lab folder'
raw=pd.read_csv(ROOT/'data/telemetry_raw.csv')
labels=pd.read_csv(ROOT/'data/training_labels.csv')
print(raw.shape)
raw.head()

## 1. Audit
Do not fix anything yet. Identify missing values, duplicated packets and mixed units. Why is keeping the first row of a shuffled duplicate risky?

In [ ]:
display(raw.isna().sum().sort_values(ascending=False))
print('Repeated packet rows:',raw.packet_id.duplicated().sum())
display(raw.temperature_unit.value_counts(dropna=False))
display(raw.pressure_unit.value_counts(dropna=False))
display(raw.quality_flag.value_counts(dropna=False))

## 2. Class imbalance
Accuracy can be high while every critical case is missed. Calculate the always-Normal accuracy, then explain why Critical recall and macro-F1 matter.

In [ ]:
display(labels.health_priority.value_counts())
labels.health_priority.value_counts().plot.bar(title='Fictional outcomes');plt.show()
print('Always-Normal accuracy among eligible label classes:',labels.loc[labels.health_priority.isin(['Normal','Suspect','Critical']),'health_priority'].eq('Normal').mean())

## 3. Time splitting and leakage
Read the split plan. We purge the last 120 minutes of every partition. Do not use `next_failure_at_utc`, `next_failure_mode`, machine IDs or target labels as features. Prepared splits are supplied so you can first learn training, then rebuild cleaning yourself.

In [ ]:
import json
print(json.dumps(json.loads((ROOT/'data/split_plan.json').read_text()),indent=2))
train=pd.read_csv(ROOT/'data/train.csv')
display(train.head())

## 4. Train and validate
The script fits preprocessing only on training rows, compares three models using validation macro-F1, freezes the choice, then evaluates the test period. This dataset was inspected in prior exercises; treat it as practice, not a fresh blind benchmark.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'train.py')],check=True)

## 5. Understand mistakes
Rows in the matrix are actual outcomes; columns are predicted outcomes. Count Critical→Normal misses. Are the results good enough for real factory alerts? Why or why not?

In [ ]:
report=json.loads((ROOT/'artifacts/evaluation.json').read_text())
print('Selected:',report['selected_model'])
display(pd.DataFrame(report['test']['classification_report']).T)
from IPython.display import Image,display
display(Image(filename=str(ROOT/'artifacts/confusion-matrix.png')))

## 6. Deploy locally
In a separate VS Code terminal, start `python -m uvicorn serve:app --host 127.0.0.1 --port 8010`. Open http://127.0.0.1:8010/docs and test `/predict` with `artifacts/sample-request.json`. The practice API has no connection to FactoryPulse or MySQL. Stop with Ctrl+C.